# Learnings from this project

## Tools
* ResNet18 - image classification

## Reading wish list
* [CS231n backpropagation notes](https://cs231n.github.io/optimization-2/)
* [micrograd's source](https://github.com/karpathy/micrograd)

* Nielsen, Neural Networks and Deep Learning, Chapter 2 (http://neuralnetworksanddeeplearning.com/chap2.html)
* [CNN wikipedia page](https://en.wikipedia.org/wiki/Convolutional_neural_network)
* [read this last](https://visionbook.mit.edu/backpropagation.html)




## Workflow

1. Download data from EuroSAT
    * Data consists of satelite images
1. Split data into training, testing, validation
1. Build transformations
    * the transformations do not operate on the image itself; when the image is sent to the model, the image is transformed that pass through


### lifecycle of one image during training

1. The DataLoader asks the dataset for image #4823.
2. The dataset opens that JPEG from disk and decodes it — now there's a 64×64 array in RAM.
3. It calls the transform on that array. Resize produces a new 224×224 array. Normalize produces another array. Each step returns a new object; the previous one becomes garbage.
4. The dataset returns that final array plus the label.
5. The DataLoader does this 32 times and stacks the results into one array of shape (32, 3, 224, 224).
6. That batch goes into the model. Loss is computed. Weights are updated.
7. The batch is discarded. Nothing about it survives.
8. Next batch: read from disk again, decode again, transform again.

so the weights do keep the updated values based on the fxns and equations, but we dont need to keep the actual information once the weights are adjusted; teh weights are what tune the model

#### `DataLoader`

**What it is:**
* sits on top of a dataset and does:
    * what order to draw in indicies
    * groups batch size based in input parameters
    * uses pythons standard iteration protocol, so the object can be looped over
        * `for images, labels in train_loader`
        * each pass includes:
            * **images** as the tensor of shape `(32, 3, 224, 224)`
            * **labels** as tensor `(32,)`
        * the `for images, labels in train_loader` is the loop that calls each batch, but within the loader there are other loops happening per image that i do not need to make myself
* `collating` == grouping; 
    * the `for loop` i write only ever references each coallated group
    * call `get item` batch size times
    * `shuffle = True` so that we get a mix of folders when building the batches
    * ? `batches are assembled lazily` - what does this term mean? ive heard it before
        * computation deferred until a result is actually needed
        * as opposed to `eager`
        * desde claude: Concretely here: DataLoader doesn't precompute and hold all batches in memory when it's constructed. It only builds a batch — reading files from disk, running the transform chain, collating — at the moment the training loop asks for the next one via for images, labels in train_loader. The next batch doesn't exist yet until that iteration requests it; the one before it is discarded once used. The opposite ("eager") would be reading and transforming all 27,000 images into memory at construction time, before any training loop even starts.
* has a `num_workers` argument; i should look into this


#### Dataset class

[The Docs](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html#creating-a-custom-dataset-for-your-files)

Their template for making a custom dataset

``` python
import os
import pandas as pd
from torchvision.io import decode_image

class CustomImageDataset(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.img_labels = pd.read_csv(annotations_file)
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = decode_image(img_path)
        label = self.img_labels.iloc[idx, 1]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label
```

* i need to create a dataset class specifically for the eurosat datasets
    * inherit `Dataset` class from `torch.utils.data.Dataset`
* this class needs to produce a dataset of tuples
* construction & setup happen together:
    * construction: 
        * use the indicies i created earlier to read certain records using the load_data() method - may require a tweak to the load data method to get specific images given indicies / filepaths
        * load the images with the given transformations
        * property: filepath of the og image
        * property: image category
        * **required by DataLoader**: len() - length of dataset
    * setup: 
        * `__getitem__` : required by all iterable python objects
        * the output needs to be a tensor (3, 224, 224) that is already transformed.
        * get_filepath() -> returns the filepath of the og image
        * get_strata() -> returns the image category


#### ResNet18 Model

* final layer of the model is referenced by `model.fc`
    * fc = fully connected
    * this is a feature extraction problem
    * feature vector mapped to output scores (one per class); ImageNet is a dataset with 1000 different classes and was used to train `ResNet`. However, this problem is looking for 10 classes, **so we have to remove the final layer and recreate it.**
    * we recreate that final layer throught training to the 10 classes we are looking for. we keep all of the relevant model information from `resnet` because we keep the bulk of the model. the last layer that maps the bulk of the model to the 1000 ImageNet classes is ok to toss becasue those 1000 classes have little to do with our 10 classes.
    * to hold the rest of the model still, we **freeze** it
    * once the new 10 categories are made, we can tweak the rest of the model if we want
    * model.parameters() - iterate over every model parameter
    * model.fc.parameters() - iterate over new layer parameters


#### Training process

* epoch: running over each batch in the dataloader object
* 3 epochs with a frozen backbone to ensure that the weights we want dont adjust too much from random noise
* unfreeze backbone
* 5 epochs 

#### What happens within an epoch

1. Forward pass
    * compute logits
    * pytorch records every operation performed on tensors that are not frozen
        * no graph will be built for frozen tensors
1. Compute Loss
    * how "wrong" a model's predictions are for a batch compared to true lanels
    * logits vs labels; higher loss score is a worse prediction
1. Backward pass
    * Computes gradient of the loss wrt every trainable weight in the network
        * i.e. if you change one weight, how much does the **loss** change and in what direction
        * gradient measures sensitivity of changes in the weight to the loss
    * graph in reverse (wrt forward pass) 
    * computes chain rule at each step to determine how much each weight contributed the final loss value
        * keep in mind: gradients accumulate by default, so call `optimizer.zero_grad()` before each batch's backward pass
    * results stored in weight.grad attribute
    * `backward()` does not update the weights, just computes and stores gradients
1. Update
    * `optimizer.step()` applies the update to every parameter that has a gradient
        * reads each parameters `.grad` that was computed by `backward()` and adjusts the parameters value using approximately: `weight -= learning_rate * weight.grad`
    * only updates the final layer (the 10 strata we care about)

#### Validation
* keep in mind, random-guess baseline for 10 classes $ln(10) ≈ 2.3$
* we care about 4 things:
    * accuracy -- a summary statistic of the count of matches between predicted class and true label
        * $correct / total$
        * can be a summary stat of the confusion matrix
    * total loss
    * confusion matrix
        * predicted $x_1$ but it should have been $x_2$
        * created with test results, not throughout training/validation
    * the specific images that were misclassified
* the classic overfitting signature is specifically train loss still falling while val loss rises — a growing gap between the two

#### confusion matrix
* compute from matrix
    * Precision = (correct predictions of C) ÷ (all predictions the model labeled C) — the diagonal cell for C divided by the sum of C's column. Answers: "when the model said C, how often was it actually right?"
    * Recall = (correct predictions of C) ÷ (all instances that were actually C) — the diagonal cell for C divided by the sum of C's row. Answers: "of all the real C images, how many did the model catch?"

#### optimization

* print(torch.backends.mps.is_available())
    * python command returns true on this device
    * metal performance shaders; model / tensor is moved onto gpu rather than on cpu